# Late Interaction Retrieval - 03: Retrieval, measured

> **MLCourse - Agentic AI - Advanced RAG - Module 15**

Notebook 02 built MaxSim and scored one query against nine passages. That is
a demo, not evidence. This notebook builds a small labelled benchmark - a
corpus, a question set, and a known-correct answer for each question - and
measures **single-vector retrieval against late interaction** on it.

### What you will learn

1. How to build a tiny labelled retrieval benchmark.
2. Recall@k and MRR, the two metrics that matter for a first-stage retriever.
3. A measured comparison of dense vs MaxSim retrieval.
4. How to read a result that **does not** show the improvement you hoped for.

### Setting expectations before we measure

Say the prediction out loud first, so we cannot retrofit the story afterwards.

A real ColBERT model is trained so its token embeddings work under MaxSim. We
are borrowing token embeddings from `all-MiniLM-L6-v2`, a bi-encoder trained
to make its *pooled* vector good. The token vectors are a by-product.

So the honest prediction is: **MaxSim over borrowed embeddings may or may not
beat the pooled baseline.** If it wins, that is a nice bonus and not proof
that ColBERT works. If it loses, that is not proof that late interaction
fails. What this notebook can genuinely demonstrate is the *mechanism* and the
*measurement method* - and we will report whatever number comes out.

Still no LLM calls and no API key.

### Setup and the labelled benchmark


In [ ]:
import time
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

MODEL = SentenceTransformer("all-MiniLM-L6-v2")
DIM = MODEL.get_embedding_dimension()

# A 24-passage corpus. Each question below has exactly ONE correct passage,
# and the corpus is seeded with near-miss distractors on purpose: passages on
# the same topic that do not answer the question, and passages that share
# vocabulary in a different sense.
CORPUS = [
    # --- capitals cluster (0-5) ---
    "Paris is the capital and most populous city of France, on the river Seine.",
    "France is a country in Western Europe with several overseas territories.",
    "The capital of Italy is Rome, a city famous for the Colosseum and the Forum.",
    "Berlin became the capital of a reunified Germany in 1990.",
    "Many capital cities grew up around a river crossing or a defensible hill.",
    "French cuisine is celebrated worldwide; Lyon is often called its capital.",
    # --- python cluster (6-11) ---
    "In Python, a list comprehension builds a new list from an iterable in one expression.",
    "Python dictionaries preserve insertion order as of version 3.7.",
    "A Python generator uses yield to produce values lazily without building a list.",
    "The Python interpreter compiles source to bytecode before executing it.",
    "Python was created by Guido van Rossum and first released in 1991.",
    "Pythons are large non-venomous snakes that kill prey by constriction.",
    # --- photosynthesis / biology cluster (12-17) ---
    "Photosynthesis converts light energy into glucose inside chloroplasts.",
    "Chlorophyll is the green pigment that absorbs light for photosynthesis.",
    "Cellular respiration breaks down glucose to release energy as ATP.",
    "Mitochondria are the organelles where cellular respiration takes place.",
    "Plants take in carbon dioxide through small pores called stomata.",
    "The light-independent reactions are also known as the Calvin cycle.",
    # --- networking cluster (18-23) ---
    "TCP guarantees ordered, reliable delivery by retransmitting lost packets.",
    "UDP sends datagrams without any delivery guarantee or ordering.",
    "DNS translates human-readable domain names into IP addresses.",
    "HTTPS encrypts HTTP traffic using TLS to prevent eavesdropping.",
    "A firewall filters network traffic according to a configured rule set.",
    "Latency is the delay before a transfer of data begins following an instruction.",
]

# (question, index of the ONE correct passage)
QUESTIONS = [
    ("what is the capital city of France", 0),
    ("which organelle performs cellular respiration", 15),
    ("how do plants absorb carbon dioxide", 16),
    ("what does DNS do", 20),
    ("which protocol does not guarantee delivery", 19),
    ("how does Python produce values lazily", 8),
    ("who created the Python language", 10),
    ("what pigment absorbs light in plants", 13),
    ("what encrypts web traffic", 21),
    ("another name for the light-independent reactions", 17),
]

print(f"corpus    : {len(CORPUS)} passages")
print(f"questions : {len(QUESTIONS)}, each with exactly one correct passage")
print(f"model     : all-MiniLM-L6-v2 ({DIM} dims)")


### 1. The two metrics

For a **first-stage retriever** the question is not "was the ranking perfect"
but "did the right passage make it into the shortlist we hand downstream".
Two metrics capture that:

- **Recall@k** - was the correct passage anywhere in the top *k*? This is the
  metric that actually matters, because a passage that misses the shortlist
  can never be recovered by a reranker later. (See
  [`11_reranking`](../11_reranking/README.md), which fixes *ordering* within
  the shortlist and can do nothing about what is absent from it.)
- **MRR (Mean Reciprocal Rank)** - the average of `1/rank` of the correct
  passage. Rank 1 scores 1.0, rank 2 scores 0.5, rank 4 scores 0.25. It
  rewards putting the answer *near the top*, not merely inside the list.

Recall@k is the go/no-go metric. MRR is the quality-of-ordering metric.

### The two retrievers


In [ ]:
def normalize(mat):
    """L2-normalise every row so a dot product equals cosine similarity."""
    arr = mat.detach().cpu().numpy() if isinstance(mat, torch.Tensor) else np.asarray(mat)
    arr = arr.astype(np.float32)
    return arr / (np.linalg.norm(arr, axis=-1, keepdims=True) + 1e-12)


# ---------- Retriever A: single-vector (bi-encoder) ----------
# One pooled vector per passage. This is ordinary vector search.
doc_pooled = MODEL.encode(CORPUS, normalize_embeddings=True)


def retrieve_dense(question):
    q = MODEL.encode(question, normalize_embeddings=True)
    return doc_pooled @ q                      # (n_docs,) cosine scores


# ---------- Retriever B: late interaction (MaxSim) ----------
# One matrix per passage. Built ONCE, offline -- exactly like the bi-encoder
# index above. This is the property that makes late interaction a retriever
# rather than a reranker.
doc_token_mats = [normalize(m) for m in MODEL.encode(CORPUS, output_value="token_embeddings")]

MAX_TOK = max(m.shape[0] for m in doc_token_mats)
DOC_PADDED = np.zeros((len(CORPUS), MAX_TOK, DIM), dtype=np.float32)
DOC_MASK = np.zeros((len(CORPUS), MAX_TOK), dtype=bool)
for i, m in enumerate(doc_token_mats):
    DOC_PADDED[i, : m.shape[0]] = m
    DOC_MASK[i, : m.shape[0]] = True


def retrieve_maxsim(question):
    q = normalize(MODEL.encode(question, output_value="token_embeddings"))
    sim = np.einsum("ik,djk->dij", q, DOC_PADDED)        # (docs, q_tok, d_tok)
    sim = np.where(DOC_MASK[:, None, :], sim, -np.inf)   # mask before the max!
    return sim.max(axis=2).sum(axis=1)                   # (n_docs,)


print(f"dense index : {doc_pooled.shape}  = {doc_pooled.size:,} floats")
print(f"maxsim index: {DOC_PADDED.shape} = {int(DOC_MASK.sum()) * DIM:,} floats (real tokens only)")
print(f"\nlate-interaction index is {int(DOC_MASK.sum()) * DIM / doc_pooled.size:.1f}x larger")


### Note what just happened in that cell

Both indexes were built **before any question arrived**. That is the whole
argument for late interaction over cross-encoder reranking: the expensive
encoding is offline in both cases, and the query-time work is arithmetic.

A cross-encoder cannot do this. It has nothing to precompute.

The last line already previews notebook 04's topic: the same 24 passages cost
roughly an order of magnitude more to store.

### Scoring both retrievers on the labelled question set


In [ ]:
def evaluate(retrieve_fn, name, k_values=(1, 3, 5)):
    """Run every question through a retriever and compute Recall@k and MRR."""
    ranks = []
    for question, gold in QUESTIONS:
        scores = retrieve_fn(question)
        order = np.argsort(-scores)                       # best first
        rank = int(np.where(order == gold)[0][0]) + 1     # 1-based rank of gold
        ranks.append(rank)

    ranks = np.array(ranks)
    result = {"name": name, "ranks": ranks, "mrr": float((1.0 / ranks).mean())}
    for k in k_values:
        result[f"recall@{k}"] = float((ranks <= k).mean())
    return result


t0 = time.perf_counter()
dense_res = evaluate(retrieve_dense, "single-vector (dense)")
dense_time = time.perf_counter() - t0

t0 = time.perf_counter()
maxsim_res = evaluate(retrieve_maxsim, "late interaction (MaxSim)")
maxsim_time = time.perf_counter() - t0

print(f"{'retriever':<28} {'R@1':>6} {'R@3':>6} {'R@5':>6} {'MRR':>7}")
print("-" * 56)
for r in (dense_res, maxsim_res):
    print(f"{r['name']:<28} {r['recall@1']:>6.2f} {r['recall@3']:>6.2f} "
          f"{r['recall@5']:>6.2f} {r['mrr']:>7.3f}")

print(f"\nMRR delta: {maxsim_res['mrr'] - dense_res['mrr']:+.3f}")
print(f"R@1 delta: {maxsim_res['recall@1'] - dense_res['recall@1']:+.2f}")


In [4]:
# ---- Per-question ranks: where did each retriever put the correct passage? -

print(f"{'question':<50} {'dense':>6} {'maxsim':>7}  verdict")
print("-" * 78)
for (question, gold), rd, rm in zip(QUESTIONS, dense_res["ranks"], maxsim_res["ranks"]):
    if rm < rd:
        verdict = "maxsim better"
    elif rm > rd:
        verdict = "dense better"
    else:
        verdict = "tie"
    print(f"{question[:48]:<50} {rd:>6} {rm:>7}  {verdict}")

wins = int((maxsim_res["ranks"] < dense_res["ranks"]).sum())
losses = int((maxsim_res["ranks"] > dense_res["ranks"]).sum())
ties = len(QUESTIONS) - wins - losses
print("-" * 78)
print(f"maxsim better on {wins}, worse on {losses}, tied on {ties} of {len(QUESTIONS)} questions")

question                                            dense  maxsim  verdict
------------------------------------------------------------------------------
what is the capital city of France                      1       1  tie
which organelle performs cellular respiration           1       1  tie
how do plants absorb carbon dioxide                     1       1  tie
what does DNS do                                        1       1  tie
which protocol does not guarantee delivery              2       2  tie
how does Python produce values lazily                   1       1  tie
who created the Python language                         1       1  tie
what pigment absorbs light in plants                    1       1  tie
what encrypts web traffic                               1       1  tie
another name for the light-independent reactions        1       1  tie
------------------------------------------------------------------------------
maxsim better on 0, worse on 0, tied on 10 of 10 question

### 2. Reading the result

**The two retrievers tied on all ten questions.** Identical ranks, identical
Recall@k, identical MRR - an MRR delta of exactly `+0.000`. Both put the
correct passage first nine times out of ten and second once.

That is the honest headline, and it is the result to sit with rather than
explain away.

### Why the tie is uninformative rather than damning

The benchmark is **saturated**. Recall@3 is 1.00 for both retrievers: every
correct passage is already in the top 3 in every case. There is no headroom
left for a better retriever to demonstrate anything. With 24 candidates and a
question set whose vocabulary overlaps its gold passage heavily, the task is
simply too easy to separate two competent methods.

This is the single most common way retrieval evaluations mislead people. A
saturated benchmark reports "no difference" for **every** technique you try,
including the ones that would help enormously at scale. The fix is not a
better retriever - it is a harder benchmark: more passages, more distractors,
and queries that share little vocabulary with their answers. See
[`10_rag_evaluation`](../10_rag_evaluation/README.md) for how that is built
properly.

Note the arithmetic too: with 10 questions, one rank change moves MRR by up to
0.1. Even a non-zero delta here would have been within noise.

### What the tie does *not* mean

It does not mean late interaction is useless, and it does not mean it works.
This benchmark had no power to answer that question either way. What we can
say is narrower and still worth having: **the mechanism is implemented
correctly and is competitive with the baseline it should be competitive
with.** A MaxSim implementation that was subtly broken - an unmasked padding
bug, missing normalisation, the max and sum the wrong way round - would have
scored visibly *worse* than dense here. It did not.

That leaves an interesting question the aggregate hides: if both retrievers
rank identically, is MaxSim actually looking at the same evidence? The next
cell says no.

Our MaxSim sums over *every* query token, including `[CLS]`, `[SEP]`, `the`,
`is`, `what`. Those tokens match something in almost any passage, so they add
a roughly constant floor to every score - and because longer, more varied
passages offer more chances to match them well, the floor is slightly higher
for them. Real ColBERT reduces this by training the model for the task and by
masking punctuation. Let's measure how big the effect is here.

### How much of each score is content vs. structural filler?


In [ ]:
STOPWORDS = {"[CLS]", "[SEP]", "the", "a", "an", "is", "are", "of", "what",
             "which", "how", "does", "do", "in", "to", "for"}

question, gold = QUESTIONS[0]
q_ids = MODEL.tokenizer(question)["input_ids"]
q_labels = MODEL.tokenizer.convert_ids_to_tokens(q_ids)
q_norm = normalize(MODEL.encode(question, output_value="token_embeddings"))

sim = np.einsum("ik,djk->dij", q_norm, DOC_PADDED)
sim = np.where(DOC_MASK[:, None, :], sim, -np.inf)
per_token_best = sim.max(axis=2)                      # (docs, q_tokens)

content_mask = np.array([t not in STOPWORDS for t in q_labels])
content_score = per_token_best[:, content_mask].sum(axis=1)
filler_score = per_token_best[:, ~content_mask].sum(axis=1)
total_score = per_token_best.sum(axis=1)

print(f"Question: {question!r}")
print(f"content tokens: {[t for t, m in zip(q_labels, content_mask) if m]}")
print(f"filler tokens : {[t for t, m in zip(q_labels, content_mask) if not m]}\n")

order = np.argsort(-total_score)[:6]
print(f"{'doc':<5} {'total':>7} {'content':>9} {'filler':>8} {'filler%':>8}  passage")
print("-" * 92)
for i in order:
    pct = 100 * filler_score[i] / total_score[i]
    star = " *" if i == gold else "  "
    print(f"D{i:<4} {total_score[i]:>7.3f} {content_score[i]:>9.3f} "
          f"{filler_score[i]:>8.3f} {pct:>7.1f}%{star} {CORPUS[i][:38]}")
print("\n* = the correct passage")

print(f"\nAcross the whole corpus, filler tokens contribute "
      f"{100 * filler_score.mean() / total_score.mean():.1f}% of the average score.")


### What that filler measurement shows

**Roughly two thirds of every MaxSim score is filler.** Structural and
function tokens - `[CLS]`, `[SEP]`, `what`, `is`, `the`, `of` - contribute
about 68% of the average score, and only the remaining third is doing any
discriminative work.

Look at the per-document breakdown to see why that is corrosive. `D1`
("France is a country in Western Europe...") has the *highest* filler share of
the six, and it is a passage that answers nothing. It is being carried up the
ranking by tokens that carry no information about relevance. Meanwhile the
`content` column separates the passages cleanly and in the right order - the
signal is there, it is just diluted 2:1 by noise.

This is a genuine weakness of applying MaxSim to embeddings that were never
trained for it, and it is one of the concrete things a real ColBERT training
run fixes. It also suggests an obvious cheap experiment: sum over content
tokens only, and see whether removing the dilution recovers the signal.

### A cheap improvement: drop filler tokens from the query side


In [ ]:
def retrieve_maxsim_content_only(question):
    """MaxSim, but summing only over CONTENT query tokens."""
    labels = MODEL.tokenizer.convert_ids_to_tokens(MODEL.tokenizer(question)["input_ids"])
    keep = np.array([t not in STOPWORDS for t in labels])
    if not keep.any():          # degenerate query of pure stopwords
        keep = np.ones(len(labels), dtype=bool)

    q = normalize(MODEL.encode(question, output_value="token_embeddings"))[keep]
    sim = np.einsum("ik,djk->dij", q, DOC_PADDED)
    sim = np.where(DOC_MASK[:, None, :], sim, -np.inf)
    return sim.max(axis=2).sum(axis=1)


content_res = evaluate(retrieve_maxsim_content_only, "MaxSim (content tokens only)")

print(f"{'retriever':<32} {'R@1':>6} {'R@3':>6} {'R@5':>6} {'MRR':>7}")
print("-" * 60)
for r in (dense_res, maxsim_res, content_res):
    print(f"{r['name']:<32} {r['recall@1']:>6.2f} {r['recall@3']:>6.2f} "
          f"{r['recall@5']:>6.2f} {r['mrr']:>7.3f}")

print("\nPer-question ranks:")
print(f"  dense        : {dense_res['ranks'].tolist()}")
print(f"  maxsim       : {maxsim_res['ranks'].tolist()}")
print(f"  maxsim/content: {content_res['ranks'].tolist()}")


### The one thing on this benchmark that did move

Dropping filler tokens from the query side took MRR from **0.950 to 1.000** -
a perfect score. The question that both other retrievers put at rank 2
("which protocol does not guarantee delivery") moved to rank 1.

Be careful about how much weight that carries. It is **one question on a
saturated ten-question benchmark**, which is exactly the size of effect this
notebook already warned is within noise. Do not leave here believing "content
masking gives you a perfect retriever."

What makes it worth reporting anyway is that it is the *only* intervention
that moved anything at all, and it moved in the direction the filler
measurement predicted before we ran it. A prediction made from a measurement
and then confirmed is weak evidence, but it is evidence - and it is a much
better reason to believe something than a number that merely came out nicely.

The mechanism is also plainly sensible: "does not guarantee delivery" is a
query whose meaning lives almost entirely in `guarantee` and `delivery`, and
whose remaining tokens (`which`, `does`, `not`) match every networking passage
equally well. Removing them removes a tie-breaker that was pure noise.

> **Practical note:** naive stopword removal is a blunt instrument and can
> backfire - `not` is a stopword in many lists and genuinely reverses meaning
> in this very query. We got away with it because MaxSim never handled the
> negation anyway. Real ColBERT masks *punctuation* rather than stopwords, and
> otherwise relies on training.

### 3. The composition that actually ships

Nothing in this notebook argues you should replace your bi-encoder with a
hand-rolled MaxSim. The real production shape is a **pipeline**, and each of
the three families does the job it is best at:

```
query
  -> STAGE 1  late interaction (or hybrid BM25+dense), top ~100   -> RECALL
  -> STAGE 2  cross-encoder rerank, keep top ~5                   -> PRECISION
  -> LLM generation
```

Late interaction competes for **stage 1**. Its selling point is that it
retrieves with more precision than a single vector while remaining
precomputable, so the shortlist handed to stage 2 is better. It does **not**
replace stage 2 - a cross-encoder reading the query and document together is
still more accurate on a small candidate set, and it is cheap there.

The decision rule, stated plainly:

| Your failure mode | The fix |
|---|---|
| "The right passage was retrieved but ranked 8th." | A cross-encoder reranker - [`11_reranking`](../11_reranking/README.md). Cheaper and very effective. |
| "The right passage never appeared in the top 100 at all." | A better first stage: hybrid search, query transformation, **or late interaction**. |
| "Neither - my passages are badly chunked." | Fix the chunking first. No retriever repairs bad chunks. |

Reach for late interaction when you have measured a **first-stage recall
ceiling** and hybrid search plus query rewriting have not lifted it.

### Key takeaways

- A first-stage retriever is judged on **Recall@k** - what misses the
  shortlist can never be recovered downstream - and **MRR** for ordering.
- Both the dense index and the late-interaction index are built **offline**.
  That is what makes late interaction a *retriever*, unlike a cross-encoder.
- **Measured here: dense and MaxSim tied on all ten questions** (MRR 0.950
  each). The benchmark was **saturated** - Recall@3 was already 1.00 - so it
  had no power to separate them. A benchmark that reports "no difference" for
  everything is telling you about the benchmark, not the techniques.
- **Measured here: filler tokens contribute ~68% of every MaxSim score.**
  Summing over `[CLS]`, `the`, `is`, `what` dilutes the signal roughly 2:1.
  Restricting the sum to content tokens took MRR to 1.000 on this set - one
  question's worth of movement, consistent with the prediction, but far too
  small a sample to generalise from.
- Late interaction competes with **stage 1**, not with reranking. The two
  compose.

**Next:** `04_storage_and_latency_cost.ipynb` - the bill for all of this,
measured, plus the compression tricks real ColBERT uses to survive it.